<a href="https://colab.research.google.com/github/computacao-aplicada/lab01-eda-Rapay/blob/main/analise_qualidade_vinhos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise da Qualidade de Vinhos — EDA + Modelos de Machine Learning

**Dataset:** Wine Quality (UCI Machine Learning Repository)
**Vinhos:** Tinto (*red*) e Branco (*white*) — analisados em conjunto e comparados

---

## Objetivo do trabalho

Este notebook tem três objetivos:

1. **Análise Exploratória dos Dados (EDA):** entender as variáveis físico-químicas dos vinhos, suas distribuições e correlações.
2. **Comparação tinto × branco:** verificar diferenças entre os dois tipos de vinho.
3. **Modelagem:** construir modelos de **classificação** (vinho bom × ruim) e de **regressão** (prever a nota de 0 a 10).

As 11 variáveis preditoras são medidas físico-químicas obtidas em laboratório. A variável-alvo `quality` é uma nota de 0 a 10 dada por avaliadores sensoriais (mediana de pelo menos 3 avaliações).

> **Como usar:** basta executar todas as células em ordem (`Ambiente de execução → Executar tudo`). O notebook baixa os dados direto da UCI, então não é necessário fazer upload dos arquivos `.csv`.


## 1. Importação das bibliotecas e carregamento dos dados

Carregamos os dois conjuntos (tinto e branco) diretamente do repositório da UCI. O separador desses arquivos é o ponto e vírgula (`;`).

In [ ]:
# Bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, mean_squared_error, mean_absolute_error,
                             r2_score, roc_auc_score, roc_curve)

# Configurações visuais
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", None)
np.random.seed(42)

print("Bibliotecas importadas com sucesso.")

In [ ]:
# Carregamento direto da UCI
URL_RED   = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
URL_WHITE = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

red   = pd.read_csv(URL_RED, sep=";")
white = pd.read_csv(URL_WHITE, sep=";")

# Coluna identificando o tipo de vinho
red["type"]   = "tinto"
white["type"] = "branco"

# Conjunto unificado
df = pd.concat([red, white], ignore_index=True)

print(f"Tinto:  {red.shape[0]} amostras")
print(f"Branco: {white.shape[0]} amostras")
print(f"Total:  {df.shape[0]} amostras, {df.shape[1]} colunas")
df.head()

> **Observação:** se o download da UCI falhar (rede), descomente a célula abaixo para fazer upload manual dos arquivos `winequality-red.csv` e `winequality-white.csv` pelo painel de arquivos do Colab.

In [ ]:
# --- ALTERNATIVA: upload manual (descomente se o download acima falhar) ---
# from google.colab import files
# print("Selecione winequality-red.csv e winequality-white.csv")
# up = files.upload()
# red   = pd.read_csv("winequality-red.csv", sep=";")
# white = pd.read_csv("winequality-white.csv", sep=";")
# red["type"] = "tinto"; white["type"] = "branco"
# df = pd.concat([red, white], ignore_index=True)
# print(df.shape)

## 2. Análise Exploratória dos Dados (EDA)

### 2.1 Estrutura e qualidade dos dados

Verificamos os tipos das colunas, valores ausentes e duplicatas.

In [ ]:
df.info()

In [ ]:
# Valores ausentes
print("Valores ausentes por coluna:")
print(df.isnull().sum())

# Duplicatas
print(f"\nLinhas duplicadas: {df.duplicated().sum()}")

In [ ]:
# Estatísticas descritivas
df.describe().T

### 2.2 Distribuição da variável-alvo (`quality`)

A nota de qualidade vai de 0 a 10, mas na prática os valores se concentram entre 3 e 9. Vamos ver como ela se distribui.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição geral
sns.countplot(data=df, x="quality", ax=axes[0], palette="viridis")
axes[0].set_title("Distribuição da qualidade (geral)")
axes[0].set_xlabel("Nota de qualidade"); axes[0].set_ylabel("Contagem")

# Distribuição por tipo
sns.countplot(data=df, x="quality", hue="type", ax=axes[1],
              palette={"tinto": "#7b1f2b", "branco": "#d4c98a"})
axes[1].set_title("Distribuição da qualidade por tipo de vinho")
axes[1].set_xlabel("Nota de qualidade"); axes[1].set_ylabel("Contagem")

plt.tight_layout(); plt.show()

print("Proporção de cada nota:")
print((df["quality"].value_counts(normalize=True).sort_index() * 100).round(1))

**Interpretação:** a maior parte dos vinhos recebe notas 5 e 6 (qualidade média). Notas muito baixas (3, 4) ou muito altas (8, 9) são raras. Isso caracteriza um problema **desbalanceado**, o que é importante levar em conta na classificação.

### 2.3 Distribuição das variáveis físico-químicas

In [ ]:
features = [c for c in df.columns if c not in ["quality", "type"]]

df[features].hist(bins=30, figsize=(15, 11), color="#4c72b0", edgecolor="white")
plt.suptitle("Distribuição das variáveis físico-químicas", y=1.01, fontsize=14)
plt.tight_layout(); plt.show()

Várias variáveis (como `residual sugar`, `chlorides`, `sulphates`) apresentam forte assimetria à direita, com presença de outliers — típico de medidas químicas.

### 2.4 Matriz de correlação

Identifica quais variáveis estão mais associadas à qualidade e quais são redundantes entre si.

In [ ]:
plt.figure(figsize=(12, 9))
corr = df[features + ["quality"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Matriz de correlação"); plt.tight_layout(); plt.show()

In [ ]:
# Correlação de cada variável com a qualidade, ordenada
corr_quality = corr["quality"].drop("quality").sort_values(ascending=False)
print("Correlação com a qualidade (ordenada):")
print(corr_quality.round(3))

**Interpretação:** `alcohol` costuma ser a variável mais correlacionada positivamente com a qualidade — vinhos com maior teor alcoólico tendem a ser melhor avaliados. `volatile acidity` aparece com correlação negativa (acidez volátil em excesso prejudica o sabor).

## 3. Comparação entre vinho tinto e branco

Aqui investigamos as diferenças físico-químicas entre os dois tipos.

In [ ]:
# Médias por tipo
comparacao = df.groupby("type")[features + ["quality"]].mean().T
comparacao["diferenca"] = comparacao["branco"] - comparacao["tinto"]
comparacao.round(3)

In [ ]:
# Boxplots comparando algumas variáveis-chave entre os tipos
vars_chave = ["alcohol", "volatile acidity", "total sulfur dioxide",
              "residual sugar", "chlorides", "quality"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, var in zip(axes.flat, vars_chave):
    sns.boxplot(data=df, x="type", y=var, ax=ax,
                palette={"tinto": "#7b1f2b", "branco": "#d4c98a"})
    ax.set_title(var); ax.set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
# Qualidade média por tipo
print("Qualidade média:")
print(df.groupby("type")["quality"].agg(["mean", "median", "std"]).round(3))

**Interpretação:** os vinhos brancos tendem a ter mais `total sulfur dioxide` e `residual sugar`, enquanto os tintos apresentam maior `volatile acidity`. A qualidade média costuma ser ligeiramente superior nos brancos, mas a diferença é pequena.

## 4. Pré-processamento

### 4.1 Criação do alvo de classificação

Como a `quality` é desbalanceada e tem muitas classes, transformamos o problema em uma **classificação binária**:

- **Bom (1):** qualidade ≥ 7
- **Ruim (0):** qualidade < 7

Esse é um corte comum na literatura desse dataset.

In [ ]:
df["bom"] = (df["quality"] >= 7).astype(int)

print("Distribuição da classe binária:")
print(df["bom"].value_counts())
print()
print((df["bom"].value_counts(normalize=True) * 100).round(1).astype(str) + " %")

### 4.2 Separação de variáveis e padronização

Codificamos o tipo de vinho como variável numérica e separamos treino/teste. A padronização (`StandardScaler`) é importante para modelos lineares.

In [ ]:
# Codifica o tipo: tinto=0, branco=1
df["type_cod"] = (df["type"] == "branco").astype(int)

# Variáveis preditoras (físico-químicas + tipo)
X_cols = features + ["type_cod"]

X = df[X_cols].copy()
y_clf = df["bom"]          # alvo de classificação
y_reg = df["quality"]      # alvo de regressão

# Split estratificado para a classificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Padronização (ajustada só no treino)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste:  {X_test.shape[0]} amostras")

## 5. Modelos de Classificação (vinho bom × ruim)

Vamos comparar dois modelos:

- **Regressão Logística** — modelo linear, simples e interpretável (usa dados padronizados).
- **Random Forest** — modelo de ensemble, geralmente mais preciso (não exige padronização).

### 5.1 Regressão Logística

In [ ]:
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
logreg.fit(X_train_sc, y_train)

y_pred_lr = logreg.predict(X_test_sc)
y_proba_lr = logreg.predict_proba(X_test_sc)[:, 1]

print("=== Regressão Logística ===")
print(f"Acurácia: {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"AUC-ROC:  {roc_auc_score(y_test, y_proba_lr):.3f}\n")
print(classification_report(y_test, y_pred_lr, target_names=["Ruim", "Bom"]))

### 5.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)   # RF não precisa de padronização

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(f"Acurácia: {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"AUC-ROC:  {roc_auc_score(y_test, y_proba_rf):.3f}\n")
print(classification_report(y_test, y_pred_rf, target_names=["Ruim", "Bom"]))

### 5.3 Comparação dos modelos de classificação

In [ ]:
# Matrizes de confusão lado a lado
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (nome, yp) in zip(axes, [("Regressão Logística", y_pred_lr),
                                  ("Random Forest", y_pred_rf)]):
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm, display_labels=["Ruim", "Bom"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(nome)

plt.tight_layout(); plt.show()

In [ ]:
# Curvas ROC
plt.figure(figsize=(8, 6))
for nome, yp in [("Reg. Logística", y_proba_lr), ("Random Forest", y_proba_rf)]:
    fpr, tpr, _ = roc_curve(y_test, yp)
    auc = roc_auc_score(y_test, yp)
    plt.plot(fpr, tpr, label=f"{nome} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Aleatório")
plt.xlabel("Taxa de Falsos Positivos"); plt.ylabel("Taxa de Verdadeiros Positivos")
plt.title("Curva ROC — comparação dos classificadores")
plt.legend(); plt.tight_layout(); plt.show()

### 5.4 Importância das variáveis (Random Forest)

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=True)

plt.figure(figsize=(9, 7))
importancias.plot(kind="barh", color="#4c72b0")
plt.title("Importância das variáveis — Random Forest (classificação)")
plt.xlabel("Importância"); plt.tight_layout(); plt.show()

print("Variáveis mais importantes:")
print(importancias.sort_values(ascending=False).round(3))

## 6. Ajuste de hiperparâmetros (GridSearch)

Para melhorar o Random Forest, fazemos uma busca em grade com validação cruzada. *(Pode levar um ou dois minutos.)*

In [ ]:
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 4],
}

grid = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
    param_grid, cv=5, scoring="roc_auc", n_jobs=-1
)
grid.fit(X_train, y_train)

print("Melhores parâmetros:", grid.best_params_)
print(f"Melhor AUC (validação cruzada): {grid.best_score_:.3f}")

# Avaliação no teste
melhor_rf = grid.best_estimator_
y_proba_best = melhor_rf.predict_proba(X_test)[:, 1]
y_pred_best = melhor_rf.predict(X_test)
print(f"\nAUC no teste:      {roc_auc_score(y_test, y_proba_best):.3f}")
print(f"Acurácia no teste: {accuracy_score(y_test, y_pred_best):.3f}")

## 7. Modelos de Regressão (prever a nota de 0 a 10)

Agora tratamos a `quality` como um valor numérico contínuo. Comparamos:

- **Regressão Linear** (baseline)
- **Random Forest Regressor**

Métricas: **RMSE** (erro quadrático médio), **MAE** (erro absoluto médio) e **R²**.

In [ ]:
# Split para regressão (mesmo X, alvo contínuo)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

# Padronização para a regressão linear
scaler_r = StandardScaler()
Xr_train_sc = scaler_r.fit_transform(Xr_train)
Xr_test_sc  = scaler_r.transform(Xr_test)

In [ ]:
def avaliar_regressao(nome, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"=== {nome} ===")
    print(f"RMSE: {rmse:.3f}")
    print(f"MAE:  {mae:.3f}")
    print(f"R²:   {r2:.3f}\n")
    return rmse, mae, r2

# Regressão Linear
linreg = LinearRegression()
linreg.fit(Xr_train_sc, yr_train)
pred_lin = linreg.predict(Xr_test_sc)
res_lin = avaliar_regressao("Regressão Linear", yr_test, pred_lin)

# Random Forest Regressor
rfr = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rfr.fit(Xr_train, yr_train)
pred_rfr = rfr.predict(Xr_test)
res_rfr = avaliar_regressao("Random Forest Regressor", yr_test, pred_rfr)

In [ ]:
# Gráfico: valores reais x previstos (Random Forest)
plt.figure(figsize=(7, 7))
plt.scatter(yr_test, pred_rfr, alpha=0.3, color="#4c72b0")
plt.plot([yr_test.min(), yr_test.max()],
         [yr_test.min(), yr_test.max()], "r--", lw=2, label="Previsão perfeita")
plt.xlabel("Qualidade real"); plt.ylabel("Qualidade prevista")
plt.title("Valores reais × previstos — Random Forest Regressor")
plt.legend(); plt.tight_layout(); plt.show()

## 8. Conclusões

**Análise exploratória**
- O dataset combina 1.599 vinhos tintos e 4.898 brancos, totalizando cerca de 6.500 amostras.
- A qualidade se concentra nas notas 5 e 6, configurando um problema desbalanceado.
- A variável `alcohol` é a mais associada positivamente à qualidade; `volatile acidity` é a mais associada negativamente.

**Comparação tinto × branco**
- Os brancos tendem a ter mais açúcar residual e dióxido de enxofre; os tintos, maior acidez volátil.
- A diferença média de qualidade entre os tipos é pequena.

**Classificação (bom × ruim)**
- O **Random Forest** superou a Regressão Logística tanto em acurácia quanto em AUC-ROC.
- As variáveis mais importantes para distinguir bons vinhos foram `alcohol`, `density` e `volatile acidity`.
- O ajuste de hiperparâmetros trouxe uma pequena melhora adicional.

**Regressão**
- O Random Forest Regressor obteve menor erro (RMSE/MAE) e maior R² que a Regressão Linear.
- Ainda assim, o R² moderado mostra que prever a nota exata é difícil — a avaliação sensorial tem um componente subjetivo que as variáveis físico-químicas não capturam totalmente.

**Considerações finais**
- O desbalanceamento limita o desempenho na classe "bom"; técnicas como *oversampling* (SMOTE) ou ajuste de limiar poderiam ser exploradas em trabalhos futuros.
- Modelos como Gradient Boosting (XGBoost, LightGBM) tendem a melhorar ainda mais os resultados.
